In [25]:
import pandas as pd

df = pd.read_csv("../data/raw/chicago_crimes.csv", nrows=100000)

df = df[
    [
        "Date",
        "Primary Type",
        "Latitude",
        "Longitude"
    ]
]

df = df.dropna(subset=["Latitude", "Longitude"])

df["Date"] = pd.to_datetime(df["Date"])

df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month

print(df.shape)

df.head()


(90895, 6)


,Date,Primary Type,Latitude,Longitude,year,month
2,2020-08-10 09:45:00,ROBBERY,41.908418,-87.677407,2020,8
4,2023-09-06 17:00:00,CRIMINAL DAMAGE,41.886018,-87.633938,2023,9
5,2023-09-06 11:00:00,THEFT,41.871835,-87.626151,2023,9
6,2019-05-21 08:20:00,BURGLARY,41.856547,-87.695605,2019,5
7,2021-07-07 10:30:00,SEX OFFENSE,41.655116,-87.594883,2021,7


In [26]:
grid_size = 0.01

df["lat_bin"] = (df["Latitude"] / grid_size).round() * grid_size
df["lon_bin"] = (df["Longitude"] / grid_size).round() * grid_size

df.head()

,Date,Primary Type,Latitude,Longitude,year,month,lat_bin,lon_bin
2,2020-08-10 09:45:00,ROBBERY,41.908418,-87.677407,2020,8,41.91,-87.68
4,2023-09-06 17:00:00,CRIMINAL DAMAGE,41.886018,-87.633938,2023,9,41.89,-87.63
5,2023-09-06 11:00:00,THEFT,41.871835,-87.626151,2023,9,41.87,-87.63
6,2019-05-21 08:20:00,BURGLARY,41.856547,-87.695605,2019,5,41.86,-87.70
7,2021-07-07 10:30:00,SEX OFFENSE,41.655116,-87.594883,2021,7,41.66,-87.59


In [27]:
crime_grid = (
    df.groupby(
        ["lat_bin", "lon_bin", "year", "month"]
    )
    .size()
    .reset_index(name="crime_count")
)

crime_grid.head()

,lat_bin,lon_bin,year,month,crime_count
0,36.62,-91.69,2001,1,1
1,36.62,-91.69,2001,3,1
2,36.62,-91.69,2001,5,1
3,36.62,-91.69,2001,7,1
4,36.62,-91.69,2001,8,1


In [28]:
threshold = crime_grid["crime_count"].quantile(0.75)

crime_grid["hotspot"] = (
    crime_grid["crime_count"] >= threshold
).astype(int)

crime_grid.head()

,lat_bin,lon_bin,year,month,crime_count,hotspot
0,36.62,-91.69,2001,1,1,0
1,36.62,-91.69,2001,3,1,0
2,36.62,-91.69,2001,5,1,0
3,36.62,-91.69,2001,7,1,0
4,36.62,-91.69,2001,8,1,0


In [29]:
crime_grid["hotspot"].value_counts()

hotspot
0    9486
1    3351
Name: count, dtype: int64

In [30]:
features = [
    "lat_bin",
    "lon_bin",
    "year",
    "month"
]

X = crime_grid[features]
y = crime_grid["hotspot"]

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [32]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [33]:
from sklearn.metrics import classification_report

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.93      0.95      0.94      1898
           1       0.85      0.79      0.82       670

    accuracy                           0.91      2568
   macro avg       0.89      0.87      0.88      2568
weighted avg       0.91      0.91      0.91      2568



In [34]:
crime_grid["hotspot"].value_counts()

hotspot
0    9486
1    3351
Name: count, dtype: int64

In [35]:
crime_grid["risk_score"] = model.predict_proba(X)[:, 1]

In [36]:
import folium

In [37]:
m = folium.Map(
    location=[41.8781, -87.6298],
    zoom_start=11
)

In [38]:
sample_data = crime_grid.sample(1000)

for _, row in sample_data.iterrows():

    if row["risk_score"] > 0.7:
        color = "red"
    elif row["risk_score"] > 0.4:
        color = "orange"
    else:
        color = "green"

    folium.CircleMarker(
        location=[row["lat_bin"], row["lon_bin"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.6,
        popup=f"Risk Score: {row['risk_score']:.2f}"
    ).add_to(m)

In [40]:
m


In [41]:
m.save("../outputs/hotspot_map.html")